In [27]:
import pandas as pd
from sqlalchemy import create_engine


POSTGRES_USER="pipeline"
POSTGRES_PASSWORD="pipeline_secret"
POSTGRES_DB="mandera_warehouse"
POSTGRES_HOST="localhost"  # Use localhost when running inside Docker
POSTGRES_PORT=5433  # Use the external port mapping from docker-compose.yml


POSTGRES_CONFIG = {
    "host":POSTGRES_HOST,
    "port": int(POSTGRES_PORT),
    "database":POSTGRES_DB,   
    "user":POSTGRES_USER,
    "password":POSTGRES_PASSWORD
}


POSTGRES_URL = (
    f"postgresql://{POSTGRES_CONFIG['user']}:{POSTGRES_CONFIG['password']}"
    f"@{POSTGRES_CONFIG['host']}:{POSTGRES_CONFIG['port']}"
    f"/{POSTGRES_CONFIG['database']}"
)

In [28]:
engine = create_engine(POSTGRES_URL)

In [42]:
from sqlalchemy import text

def transform(engine_value):
    """
    Read all customers from raw, deduplicate, clean, and upsert to staging.
    """
    with engine_value.connect() as conn:

        result = conn.execute(text('SELECT * FROM raw.customers_raw'))
        #print("  ✅ Fetched raw customers data")
        #print(f"  🧮 Number of raw customers fetched: {result.rowcount}")

        df = pd.DataFrame(result.fetchall(), columns=result.keys())
        #print(f"  🧮 Number of raw customers in DataFrame: {len(df)}")
        #print(df.head())
        
        return df

    if df.empty:
        print("  ⊘ No customer data to transform")
        return None

In [41]:
df_customers = transform(engine)

In [43]:
df_customers.head()

,customer_id,name,email,phone,city,batch_id,created_at
0,CUST82404,Laura Manning,hollyrice@example.net,397.928.3089x4988,Arthurstad,2026_04_07_21_batch_02,2026-04-07 21:16:03.295
1,CUST46895,Nicole Bruce,jacksonmaria@example.org,892-922-5958,Port Angela,2026_04_07_21_batch_02,2026-04-07 21:16:03.296
2,CUST57790,Luis Turner,cameronallen@example.com,(209)275-3033,Brownmouth,2026_04_07_21_batch_02,2026-04-07 21:16:03.296
3,CUST53096,Sarah Davis,mcox@example.com,001-230-376-2467x07234,Amyland,2026_04_07_21_batch_02,2026-04-07 21:16:03.296
4,CUST95193,John Berry,ramosrichard@example.com,(912)571-0476x550,Port Abigailtown,2026_04_07_21_batch_02,2026-04-07 21:16:03.296


In [48]:
df_customers["created_at"] = pd.to_datetime(df_customers["created_at"], errors="coerce")
df_customers.rename(columns={"created_at": "created_datetime"}, inplace=True)

In [49]:
df_customers.head()

,customer_id,name,email,phone,city,batch_id,created_datetime
0,CUST82404,Laura Manning,hollyrice@example.net,397.928.3089x4988,Arthurstad,2026_04_07_21_batch_02,2026-04-07 21:16:03.295
1,CUST46895,Nicole Bruce,jacksonmaria@example.org,892-922-5958,Port Angela,2026_04_07_21_batch_02,2026-04-07 21:16:03.296
2,CUST57790,Luis Turner,cameronallen@example.com,(209)275-3033,Brownmouth,2026_04_07_21_batch_02,2026-04-07 21:16:03.296
3,CUST53096,Sarah Davis,mcox@example.com,001-230-376-2467x07234,Amyland,2026_04_07_21_batch_02,2026-04-07 21:16:03.296
4,CUST95193,John Berry,ramosrichard@example.com,(912)571-0476x550,Port Abigailtown,2026_04_07_21_batch_02,2026-04-07 21:16:03.296
